In [1]:
!pip uninstall -y transformers accelerate tokenizers

!pip install -q \
transformers==4.55.4 \
accelerate==1.10.1 \
tokenizers==0.21.4 \
evaluate \
wandb

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.9 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently

In [2]:
import transformers, accelerate, tokenizers

print(transformers.__version__)
print(accelerate.__version__)
print(tokenizers.__version__)

4.55.4
1.10.1
0.21.4


In [3]:
import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd
import torch
import evaluate
import wandb

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

2026-07-22 08:55:01.884306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784710502.059268      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784710502.114158      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784710502.531781      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784710502.531825      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784710502.531827      58 computation_placer.cc:177] computation placer alr

In [4]:
MODEL_NAME = "microsoft/deberta-v3-small"

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

MAX_LENGTH = 256

BATCH_SIZE = 16

LEARNING_RATE = 2e-5

EPOCHS = 5

SEED = 42

In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [6]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb")

wandb.login(key=wandb_key)

run = wandb.init(
    project="DL-GenAI-2026-T2",
    name="deberta_v3_small",
    config={
        "model": "microsoft/deberta-v3-small",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "max_length": MAX_LENGTH,
    },
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: taransaggu (taransaggu-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
# convert answer options into numerical labels

label_to_id = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

id_to_label = {v: k for k, v in label_to_id.items()}

train_df["label"] = train_df["answer"].map(label_to_id)

In [9]:
def create_text(row):

    return (
        f"Question: {row['prompt']}\n\n"
        f"Option A: {row['A']}\n"
        f"Option B: {row['B']}\n"
        f"Option C: {row['C']}\n"
        f"Option D: {row['D']}\n"
        f"Option E: {row['E']}"
    )

train_df["text"] = train_df.apply(create_text, axis=1)
test_df["text"] = test_df.apply(create_text, axis=1)

In [10]:
# train_data, valid_data = train_test_split(
#     train_df,
#     test_size=0.2,
#     random_state=SEED,
#     stratify=train_df["label"]
# )

# print("Training Set :", train_data.shape)
# print("Validation Set :", valid_data.shape)

In [11]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, valid_idx = next(
    gss.split(
        train_df,
        groups=train_df["prompt"]
    )
)

train_data = train_df.iloc[train_idx].reset_index(drop=True)
valid_data = train_df.iloc[valid_idx].reset_index(drop=True)

In [12]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_data[["text", "label"]],
    preserve_index=False
)

valid_dataset = Dataset.from_pandas(
    valid_data[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]],
    preserve_index=False
)

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [14]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [15]:
# Tokenize datasets
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

valid_dataset = valid_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/1609 [00:00<?, ? examples/s]

Map:   0%|          | 0/391 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [16]:
print(train_dataset.column_names)

print(train_dataset[0])

['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
{'text': 'Question: What is accelerator-based light-ion fusion?\n\nOption A: Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.\nOption B: Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.\nOption C: Accelerator-based light-ion fusion is a technique that uses pa

In [17]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [18]:
accuracy_metric = evaluate.load("accuracy")

In [19]:
def map_at_3(predictions, labels):

    top3 = np.argsort(predictions, axis=1)[:, ::-1][:, :3]

    score = 0

    for true, pred in zip(labels, top3):

        if true == pred[0]:
            score += 1

        elif true == pred[1]:
            score += 0.5

        elif true == pred[2]:
            score += 1/3

    return score / len(labels)

In [20]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )["accuracy"]

    map3 = map_at_3(logits, labels)

    return {
        "accuracy": accuracy,
        "map@3": map3
    }

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
print(next(model.parameters()).dtype)

torch.float32


In [23]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [24]:
training_args = TrainingArguments(
    output_dir="./deberta_checkpoints",
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,
    save_total_limit=2,
    report_to="wandb",
    run_name="deberta_v3_small",
    fp16=True,

    seed=SEED
)

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,1.583500,1.428199,0.396419,0.586530
2,0.993500,0.537066,0.864450,0.916454
3,0.318900,0.112778,0.989770,0.993180
4,0.081600,0.031275,0.994885,0.997442
5,0.030800,0.019943,0.997442,0.998721


TrainOutput(global_step=255, training_loss=0.5903870448177936, metrics={'train_runtime': 176.1997, 'train_samples_per_second': 45.658, 'train_steps_per_second': 1.447, 'total_flos': 532897599505920.0, 'train_loss': 0.5903870448177936, 'epoch': 5.0})

In [33]:
print(train_data["label"].value_counts(normalize=True))
print(valid_data["label"].value_counts(normalize=True))

label
1    0.244873
2    0.230578
3    0.184587
0    0.178372
4    0.161591
Name: proportion, dtype: float64
label
1    0.245524
2    0.225064
0    0.209719
4    0.163683
3    0.156010
Name: proportion, dtype: float64


In [35]:
print(train_idx[:10])
print(valid_idx[:10])

[ 1  2  3  4  5  6  7  8 10 11]
[ 0  9 15 19 24 25 27 30 33 35]


In [34]:
train_data["prompt"].str.len().describe()
valid_data["prompt"].str.len().describe()

count    391.000000
mean     116.437340
std       41.084722
min       19.000000
25%       90.000000
50%      113.000000
75%      138.000000
max      315.000000
Name: prompt, dtype: float64

In [27]:
train_texts = set(train_data["text"])
print(valid_data["text"].isin(train_texts).sum())

0


In [28]:
train_prompts = set(train_data["prompt"])
print(valid_data["prompt"].isin(train_prompts).sum())

0


In [29]:
predictions = trainer.predict(test_dataset)
logits = predictions.predictions

In [30]:
top3 = np.argsort(logits, axis=1)[:, ::-1][:, :3]

predictions = []

for row in top3:
    predictions.append(" ".join(id_to_label[i] for i in row))

In [31]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission.head()

,id,prediction
0,1,A D E
1,2,B D C
2,3,B C D
3,4,E C D
4,5,C A B


In [37]:
submission.to_csv("submission.csv", index=False)

In [36]:
run.finish()

eval/accuracy,▁▆███
eval/loss,█▄▁▁▁
eval/map@3,▁▇███
eval/runtime,▁▃▇█▇
eval/samples_per_second,█▆▂▁▂
eval/steps_per_second,█▆▂▁▂
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▁▃▃▄▅▆▆███
+4,...
